In [1]:
from pathlib import Path

from katabatic.artifacts import LocalArtifactStore
from katabatic.models.tabddpm.models import Tabddpm
from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from katabatic.utils.preprocess import preprocess_dataset

ROOT = None

for p in [Path.cwd(), *Path.cwd().parents]:
    if (p / "datasets").exists() and (p / "models").exists():
        ROOT = p
        break

raw_file = ROOT / "datasets" / "shuttle.csv"
processed_file = ROOT / "preprocessed_data" / "shuttle_tabddpm.csv"
artifact_dir = ROOT / "artifacts"

print("ROOT:", ROOT)
print("Raw dataset:", raw_file)
print("Dataset exists:", raw_file.exists())

processed_file.parent.mkdir(parents=True, exist_ok=True)

preprocess_dataset(
    str(raw_file),
    str(processed_file),
    target_col="class"
)

store = LocalArtifactStore(str(artifact_dir))

pipeline = TrainTestSplitPipeline(
    model=Tabddpm()
)

pipeline._evaluations = []

results = pipeline.run(
    input_csv=str(processed_file),
    dataset_name="shuttle_tabddpm",
    artifact_store=store,
    model_name="tabddpm",
)

print(results)

ROOT: c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\katabatic
Raw dataset: c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\katabatic\datasets\shuttle.csv
Dataset exists: True
Preprocessing: c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\katabatic\datasets\shuttle.csv
Saved preprocessed dataset to: c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\katabatic\preprocessed_data\shuttle_tabddpm.csv
Loaded data with shape: (58000, 10)
Train label distribution:
 class
1    0.785970
4    0.153491
5    0.056336
3    0.002953
2    0.000862
7    0.000216
6    0.000172
Name: proportion, dtype: float64
Test label distribution:
 class
1    0.785948
4    0.153534
5    0.056293
3    0.002931
2    0.000862
7    0.000259
6    0.000172
Name: proportion, dtype: float64
Saved dataset artifact under datasets/shuttle_tabddpm/split-20260808-123106
Step 100/200 | MLoss: 0.0000 | GLoss: 4.0548
Step 200/200 | MLoss: 0.00

In [2]:
import pandas as pd

from katabatic.pipeline.evaluation_pipeline import SyntheticEvaluationPipeline

splits_root = ROOT / "artifacts" / "datasets" / "shuttle_tabddpm"

split_dirs = sorted(
    [p for p in splits_root.glob("split-*") if p.is_dir()],
    key=lambda p: p.stat().st_mtime
)

latest_split = split_dirs[-1]

print("Using split:", latest_split)

train_df = pd.read_csv(
    latest_split / "train" / "train_full.csv"
)

test_df = pd.read_csv(
    latest_split / "test" / "test_full.csv"
)

target_col = "class"

categorical_cols = []

continuous_cols = [
    "time",
    "a1",
    "a2",
    "a3",
    "a4",
    "a5",
    "a6",
    "a7",
    "a8",
]

model = pipeline.model

synthetic_df = model.sample(
    len(train_df),
    seed=42
)

print("Synthetic type:", type(synthetic_df))
print("\nSynthetic sample:")
print(synthetic_df.head())

evaluation_pipeline = SyntheticEvaluationPipeline(
    dimensions=[
        "fidelity",
        "utility",
        "diversity",
        "privacy",
        "consistency",
        "stability",
    ],
    categorical_cols=categorical_cols,
    continuous_cols=continuous_cols,
)

report = evaluation_pipeline.run(
    real_data=train_df,
    synthetic_data=synthetic_df,
    target_col=target_col,
    test_data=test_df,
    model=model,
)

print("\nComposite Score:", report.composite_score)

print("\nDimension Scores:")
for dimension, score in report.dimension_scores.items():
    print(f"{dimension}: {score}")

Using split: c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\katabatic\artifacts\datasets\shuttle_tabddpm\split-20260808-123106
Synthetic type: <class 'pandas.core.frame.DataFrame'>

Synthetic sample:
       time        a1        a2        a3        a4        a5        a6  \
0  0.539974  0.466720  0.493451  0.258213  1.010963  0.227910  0.218240   
1  0.783113  0.430881  0.880543  0.126377  0.545363  0.314509 -0.015405   
2  0.216875  0.157042  0.864523 -0.061502  0.431591  0.184191  0.330885   
3  0.754810  0.971917  0.105658  0.192520  1.258440  0.487990  0.383358   
4  0.438080  0.419913  1.036898  0.108911  1.071302  0.193448  0.550756   

         a7        a8  class  
0 -0.244839  0.427606      1  
1  1.401046  1.539187      1  
2  0.326911  0.413199      1  
3  0.944316  1.512009      1  
4  0.075950  0.275413      1  

Running fidelity evaluation...

=== Fidelity Evaluation ===
Overall fidelity score: 0.7054

Continuous Wasserstein (normalised, lower = be